<center> 
    <img src="https://rare-gallery.com/thumbs/4513600-long-hair-night-hatsune-miku-twintails-vocaloid-space.jpg">
</center>


# Theory

## 🛰️ Orbit Wars — Rule-base × ML Shot Validator Hybrid
This notebook ships a **hybrid agent** that pairs a strong public rule-based agent (the
Tamrazov × Ykhnkf line, descended from `pilkwang/structured-baseline`) with a small
**numpy-only "Shot Validator" MLP** that filters out attacks the rule-base would otherwise
make but which an ML model has learned tend to fail.

The validator is intentionally **conservative**: it only ever *rejects* shots, never proposes
new ones. Worst case it does nothing and the agent is identical to the rule-base. Best case
it removes wasteful attacks against well-defended targets and lets the agent conserve ships
for the next opening.

In local 2P play (8 seeds × 5 opponents × 2 sides = 80 games per side), the hybrid wins
**84%** vs **65%** for the rule-base alone — a **+19pp** swing driven mostly by the harder
opponent classes (tier3 +25pp, tier4 +43pp).

### 1. Why a Hybrid?

A pure ML approach (PPO from scratch, SFT distillation from teachers, multi-teacher SFT)
keeps hitting the same ceiling against the strongest public rule-based agents — win rate
against tier3+ opponents collapses to ~0% even after 1000+ PPO updates. Five separate ML
attempts ran into the same wall.

The mechanism is the usual sparse-reward trap: the network only gets a +1 / -1 signal at
end-of-game, against a strong opponent it loses almost every rollout, and the gradient
ends up pushing the policy toward defensive no-ops rather than discovering tier3-beating
strategies.

A pure rule-base approach (the path our parent notebook took) has the opposite ceiling:
it plays a coherent strategy out of the box, but every constant-tweak we tried either
helped against one opponent and hurt another, or had no measurable effect.

This notebook takes the **third path**:

```
[obs] ──► rule-base (v4 lineage) ──► candidate moves
                                     │
                                     ▼
                              ML shot validator ──► drop low-P(success) shots
                                     │
                                     ▼
                                 final action
```

The rule-base does the heavy strategic lifting. The ML model only votes *no* on individual
shots. Because the rule-base's coherent strategy is preserved, the ML doesn't need to
discover anything from scratch — it just needs to learn one local question:

> *Given this source planet, this target, and this fleet size — does this shot usually
> end with us owning the target 10 turns later?*

That question has a clean per-shot binary label, dense across every game (~180 shots / game),
and the wrong answer just leaves a v4-equivalent action in place.


### 2. Shot Validator design

#### Inputs (24-dim float32)

For every shot the rule-base proposes, we encode:

| group | features |
| --- | --- |
| **source planet** | ships, production, radius |
| **target planet** | ships, production, radius, owner one-hot (mine / neutral / enemy) |
| **shot** | ships sent, ship fraction (sent / source ships), distance, ETA in turns, computed fleet speed |
| **in-flight** | count + ship total of allied & enemy fleets |
| **meta** | turn number, my total ships, enemy total ships, ship diff, my planet count, enemy planet count |

All scalars are normalised to roughly [0, 1] so the MLP doesn't have to learn per-feature
scales.

#### Label

For each shot, walk forward in the played-out game from the expected arrival turn `t` for
`t+10` turns. Label is `1` iff the target planet's owner is *us* on any of those turns,
else `0`.

Crucially, **shots that reinforce our own planets are excluded from the dataset entirely**.
Self-reinforcement is trivially "successful" (we already own the target) and would dilute
the signal — without filtering, the positive rate is ~96%. After filtering, the positive
rate drops to **70.8%**, which leaves real negative signal for the model to learn from.

#### Model

A tiny three-layer MLP, ~5k parameters total:

```
input (24) → Linear(64) → ReLU → Linear(32) → ReLU → Linear(1) → sigmoid → P(success)
```

Trained with `BCEWithLogitsLoss(pos_weight = neg/pos)` on 8.8k training shots from
games against five different opponents, validated by *game id* (not row) to prevent
leakage between train and val.

After 40 epochs:

- val accuracy at threshold 0.5: **76.8%**
- val accuracy at threshold 0.3: **80.8%**
- mean P(positive) given true positive: **0.68**
- mean P(positive) given true negative: **0.38**

The 0.30 separation is small in absolute terms, but it doesn't need to be large — every
correctly rejected wasteful shot saves ships, and every incorrect rejection just leaves a
v4 default in place.

#### Inference: threshold gate

At inference time we rebuild the same 24-dim feature for every shot the rule-base wants to
take, run the MLP, and **drop** any shot whose predicted P(success) is below a threshold.
Self-reinforcement always passes through.

We swept four thresholds:

| threshold | local win rate |
|---|---|
| 0.2 (lenient) | 76% |
| 0.3 | 78% |
| **0.4** | **84%** ⭐ |
| 0.5 (strict) | 57% (over-rejects) |

0.4 is the sweet spot — strict enough to remove the bad tail, lenient enough not to reject
the merely uncertain.


## 3. Validation findings

### 3.1 Per-opponent win rate (8 seeds × 2 sides = 16 games / cell)

| opponent | hybrid (t=0.4) | rule-base only | Δ |
|---|---|---|---|
| `v1_sniper` | 16/16 (100%) | 16/16 (100%) | 0 |
| `v2_structured` | 13/16 (81%) | 12/16 (75%) | +6pp |
| `exp007_tier3` | **13/16 (81%)** | 9/16 (56%) | **+25pp** |
| `exp007_tier4` | **9/16 (56%)** → **13/16 (81%)** | 6/16 (38%) | **+43pp** |
| `orbitbotnext` | 11/16 (69%) → 12/16 (75%) | 9/16 (56%) | +13–19pp |
| **overall** | **67/80 (84%)** | **52/80 (65%)** | **+19pp** |

The pattern is the one the design predicted: against weak opponents (sniper) there's
nothing for the validator to do, both agents win comfortably. Against the strong opponents
where v4 alone struggles (tier4 38%), the validator's ship conservation lets the agent
trade more efficiently and the win rate roughly doubles.

### 3.2 What changes turn-by-turn

Across an average game, the validator drops on the order of **3–10%** of the rule-base's
shots. The dropped shots cluster around two patterns:

- **Late-game over-extension**: trying to capture a target that the model thinks will be
  re-taken before our reinforcement arrives.
- **Defended-target attacks with marginal ship counts**: shots where ship_fraction is high
  enough that the source becomes vulnerable, but the model has low confidence the target
  will actually fall.

In both cases the rejection conserves ships for the next turn's rule-base decision, and
the rule-base picks a better target with the saved garrison.

### 3.3 No regression vs the rule-base

Every opponent class shows hybrid ≥ rule-base. There is no opponent for which adding the
validator hurts. This is the design's core safety property: rejection-only overrides cannot
introduce a worse action than the rule-base's own choice — they can only fail to improve it.


## 4. create neccessary instruments for training (thanks, YumeNeko)

The trained MLP weights are tiny (~15 KB) so we embed them as base64 here and decode
back into a `weights.npz` file next to `submission.py` at submission time.

author: [YumeNeko](https://www.kaggle.com/kashiwaba)

In [23]:
!mkdir -p src

# Config

In [24]:
%%writefile default_cfg.yaml

seed: 42
run_name: orbit_wars_ppo
device: auto
save_dir: /kaggle/working/artifacts
checkpoint_every: 50
log_every: 1
opponent: self
self_play_update_interval: 50
self_play_deterministic: false
alternate_player_sides: true

env:
  candidate_count: 8
  ship_bucket_count: 8

model:
  hidden_size: 128

ppo:
  rollout_steps: 64
  num_envs: 2
  total_updates: 2000 # Note: For this public Notebook, total_updates is set to 100 to keep runtime short. For full training, increase it to 2000.
  epochs: 4
  minibatch_size: 256
  gamma: 0.99
  clip_coef: 0.2
  ent_coef: 0.01
  vf_coef: 0.5
  lr: 0.0003
  max_grad_norm: 0.5

Overwriting default_cfg.yaml


In [25]:
%%writefile src/__init__.py

from .config import TrainConfig, default_train_config_path, load_train_config

__all__ = ["TrainConfig", "default_train_config_path", "load_train_config"]

Overwriting src/__init__.py


In [26]:
%%writefile src/config.py

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import yaml


@dataclass(slots=True)
class EnvConfig:
    board_size: float = 100.0
    episode_steps: int = 500
    candidate_count: int = 8
    ship_bucket_count: int = 8
    max_planets: int = 48
    max_ships: float = 400.0
    max_production: float = 5.0


@dataclass(slots=True)
class ModelConfig:
    hidden_size: int = 128


@dataclass(slots=True)
class PPOConfig:
    rollout_steps: int = 32
    num_envs: int = 4
    total_updates: int = 200
    epochs: int = 4
    minibatch_size: int = 512
    gamma: float = 0.99
    clip_coef: float = 0.2
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    lr: float = 3e-4
    max_grad_norm: float = 0.5


@dataclass(slots=True)
class TrainConfig:
    seed: int = 42
    run_name: str = "orbit_wars_template_ppo"
    device: str = "auto"
    save_dir: str = "artifacts/rl_template"
    checkpoint_every: int = 10
    log_every: int = 1
    opponent: str = "random"
    self_play_update_interval: int = 10
    self_play_deterministic: bool = False
    alternate_player_sides: bool = True
    env: EnvConfig = field(default_factory=EnvConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    ppo: PPOConfig = field(default_factory=PPOConfig)


def default_train_config_path() -> Path:
    return Path(__file__).resolve().parent / "configs" / "default.yaml"


def load_train_config(path: str | Path) -> TrainConfig:
    config_path = Path(path)
    data = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
    if not isinstance(data, dict):
        raise ValueError(f"YAML config must be a mapping: {config_path}")
    return train_config_from_dict(data)


def train_config_from_dict(data: dict[str, Any]) -> TrainConfig:
    cfg = TrainConfig()
    _update_dataclass(cfg, data, skip={"env", "model", "ppo"})
    _update_dataclass(cfg.env, data.get("env", {}))
    _update_dataclass(cfg.model, data.get("model", {}))
    _update_dataclass(cfg.ppo, data.get("ppo", {}))
    return cfg


def _update_dataclass(instance: Any, values: dict[str, Any], skip: set[str] | None = None) -> None:
    if not isinstance(values, dict):
        return
    skip = skip or set()
    for key, value in values.items():
        if key in skip or not hasattr(instance, key):
            continue
        default = getattr(instance, key)
        setattr(instance, key, _coerce_value(value, default))


def _coerce_value(value: Any, default: Any) -> Any:
    if isinstance(default, bool):
        if isinstance(value, str):
            lowered = value.strip().lower()
            if lowered in {"1", "true", "yes", "on"}:
                return True
            if lowered in {"0", "false", "no", "off"}:
                return False
        return bool(value)
    if isinstance(default, int) and not isinstance(default, bool):
        return int(value)
    if isinstance(default, float):
        return float(value)
    return value


Overwriting src/config.py


In [27]:
%%writefile src/features.py

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any

import numpy as np

from .config import EnvConfig
from .game_types import GameState, PlanetState, parse_observation

BOARD_CENTER = (50.0, 50.0)
ROTATION_RADIUS_LIMIT = 50.0
SUN_RADIUS = 10.0
PLANET_LAUNCH_RADIUS_OFFSET = 0.1


@dataclass(slots=True)
class DecisionContext:
    env_index: int
    source_id: int
    candidate_ids: list[int]
    candidate_mask: np.ndarray
    ship_counts: list[int]
    target_angles: list[float]


@dataclass(slots=True)
class TurnBatch:
    self_features: np.ndarray
    candidate_features: np.ndarray
    global_features: np.ndarray
    candidate_mask: np.ndarray
    contexts: list[DecisionContext]
    state: GameState


def self_feature_dim() -> int:
    return 11


def candidate_feature_dim() -> int:
    return 14


def global_feature_dim() -> int:
    return 8


def encode_turn(
    observation: Any,
    env_cfg: EnvConfig,
    *,
    env_index: int = 0,
) -> TurnBatch:
    state = observation if isinstance(observation, GameState) else parse_observation(observation)
    my_planets = sorted((planet for planet in state.planets if planet.owner == state.player), key=lambda planet: planet.id)
    if not my_planets:
        return TurnBatch(
            self_features=np.zeros((0, self_feature_dim()), dtype=np.float32),
            candidate_features=np.zeros((0, env_cfg.candidate_count, candidate_feature_dim()), dtype=np.float32),
            global_features=np.zeros((0, global_feature_dim()), dtype=np.float32),
            candidate_mask=np.zeros((0, env_cfg.candidate_count), dtype=bool),
            contexts=[],
            state=state,
        )

    global_feat = build_global_features(state, env_cfg)
    self_rows: list[np.ndarray] = []
    candidate_rows: list[np.ndarray] = []
    candidate_masks: list[np.ndarray] = []
    contexts: list[DecisionContext] = []

    for src in my_planets:
        candidates = build_candidates(src, state, env_cfg)
        cand_feat, cand_mask, ship_counts, candidate_ids, target_angles = build_candidate_features(
            src,
            candidates,
            state,
            env_cfg,
        )
        self_rows.append(build_self_features(src, state, env_cfg))
        candidate_rows.append(cand_feat)
        candidate_masks.append(cand_mask)
        contexts.append(
            DecisionContext(
                env_index=env_index,
                source_id=src.id,
                candidate_ids=candidate_ids,
                candidate_mask=cand_mask,
                ship_counts=ship_counts,
                target_angles=target_angles,
            )
        )

    return TurnBatch(
        self_features=np.asarray(self_rows, dtype=np.float32),
        candidate_features=np.asarray(candidate_rows, dtype=np.float32),
        global_features=np.repeat(global_feat[None, :], len(self_rows), axis=0),
        candidate_mask=np.asarray(candidate_masks, dtype=bool),
        contexts=contexts,
        state=state,
    )


def build_candidates(src: PlanetState, state: GameState, env_cfg: EnvConfig) -> list[PlanetState]:
    others = [planet for planet in state.planets if planet.id != src.id]
    enemy_quota = env_cfg.candidate_count // 3
    neutral_quota = env_cfg.candidate_count // 3
    friendly_quota = env_cfg.candidate_count - enemy_quota - neutral_quota

    enemies = sorted(
        (planet for planet in others if planet.owner not in {-1, state.player}),
        key=lambda planet: (distance(src, planet), planet.id),
    )[:enemy_quota]
    neutrals = sorted(
        (planet for planet in others if planet.owner == -1),
        key=lambda planet: (distance(src, planet), planet.id),
    )[:neutral_quota]
    friendlies = sorted(
        (planet for planet in others if planet.owner == state.player),
        key=lambda planet: (distance(src, planet), planet.id),
    )[:friendly_quota]

    selected_ids = {planet.id for planet in enemies + neutrals + friendlies}
    candidates = enemies + neutrals + friendlies
    if len(candidates) >= env_cfg.candidate_count:
        return candidates[: env_cfg.candidate_count]

    fallback = sorted(
        (planet for planet in others if planet.id not in selected_ids),
        key=lambda planet: (distance(src, planet), planet.id),
    )
    candidates.extend(fallback[: env_cfg.candidate_count - len(candidates)])
    return candidates


def build_self_features(src: PlanetState, state: GameState, env_cfg: EnvConfig) -> np.ndarray:
    my_planets = [planet for planet in state.planets if planet.owner == state.player]
    enemy_planets = [planet for planet in state.planets if planet.owner not in {-1, state.player}]
    return np.asarray(
        [
            1.0,
            src.x / env_cfg.board_size,
            src.y / env_cfg.board_size,
            src.radius / 5.0,
            min(src.ships, env_cfg.max_ships) / env_cfg.max_ships,
            src.production / env_cfg.max_production,
            1.0 if is_rotating_planet(src) else 0.0,
            len(my_planets) / env_cfg.max_planets,
            len(enemy_planets) / env_cfg.max_planets,
            total_ships(my_planets) / (env_cfg.max_planets * env_cfg.max_ships),
            total_ships(enemy_planets) / (env_cfg.max_planets * env_cfg.max_ships),
        ],
        dtype=np.float32,
    )


def build_candidate_features(
    src: PlanetState,
    candidates: list[PlanetState],
    state: GameState,
    env_cfg: EnvConfig,
) -> tuple[np.ndarray, np.ndarray, list[int], list[int], list[float]]:
    features = np.zeros((env_cfg.candidate_count, candidate_feature_dim()), dtype=np.float32)
    candidate_mask = np.zeros((env_cfg.candidate_count,), dtype=bool)
    ship_counts = [0] * env_cfg.candidate_count
    candidate_ids = [-1] * env_cfg.candidate_count
    target_angles = [0.0] * env_cfg.candidate_count
    candidate_mask[0] = True

    for idx, tgt in enumerate(candidates, start=1):
        if idx >= env_cfg.candidate_count:
            break
        dx = tgt.x - src.x
        dy = tgt.y - src.y
        angle = math.atan2(dy, dx)
        crosses_sun = shot_crosses_sun(src, angle, tgt)
        ships_needed = fixed_ship_count(src, tgt)
        features[idx] = np.asarray(
            [
                1.0,
                1.0 if tgt.owner == -1 else 0.0,
                1.0 if tgt.owner == state.player else 0.0,
                1.0 if tgt.owner not in {-1, state.player} else 0.0,
                tgt.x / env_cfg.board_size,
                tgt.y / env_cfg.board_size,
                dx / env_cfg.board_size,
                dy / env_cfg.board_size,
                distance(src, tgt) / env_cfg.board_size,
                min(tgt.ships, env_cfg.max_ships) / env_cfg.max_ships,
                tgt.production / env_cfg.max_production,
                1.0 if is_rotating_planet(tgt) else 0.0,
                1.0 if crosses_sun else 0.0,
                min(src.ships, env_cfg.max_ships) / env_cfg.max_ships,
            ],
            dtype=np.float32,
        )
        ship_counts[idx] = ships_needed
        candidate_mask[idx] = ships_needed > 0 and not crosses_sun and src.ships >= ships_needed
        candidate_ids[idx] = tgt.id
        target_angles[idx] = angle

    return features, candidate_mask, ship_counts, candidate_ids, target_angles


def build_global_features(state: GameState, env_cfg: EnvConfig) -> np.ndarray:
    my_planets = [planet for planet in state.planets if planet.owner == state.player]
    enemy_planets = [planet for planet in state.planets if planet.owner not in {-1, state.player}]
    neutral_planets = [planet for planet in state.planets if planet.owner == -1]
    my_fleets = [fleet for fleet in state.fleets if fleet.owner == state.player]
    enemy_fleets = [fleet for fleet in state.fleets if fleet.owner != state.player]
    return np.asarray(
        [
            state.step / env_cfg.episode_steps,
            len(my_planets) / env_cfg.max_planets,
            len(enemy_planets) / env_cfg.max_planets,
            len(neutral_planets) / env_cfg.max_planets,
            total_ships(my_planets) / (env_cfg.max_planets * env_cfg.max_ships),
            total_ships(enemy_planets) / (env_cfg.max_planets * env_cfg.max_ships),
            sum(fleet.ships for fleet in my_fleets) / (env_cfg.max_planets * env_cfg.max_ships),
            sum(fleet.ships for fleet in enemy_fleets) / (env_cfg.max_planets * env_cfg.max_ships),
        ],
        dtype=np.float32,
    )


def fixed_ship_count(src: PlanetState, tgt: PlanetState) -> int:
    return max(tgt.ships + 1, 20)


def distance(a: PlanetState, b: PlanetState) -> float:
    return math.hypot(a.x - b.x, a.y - b.y)


def total_ships(planets: list[PlanetState]) -> float:
    return float(sum(planet.ships for planet in planets))


def is_rotating_planet(planet: PlanetState) -> bool:
    dx = planet.x - BOARD_CENTER[0]
    dy = planet.y - BOARD_CENTER[1]
    orbital_radius = math.hypot(dx, dy)
    return orbital_radius + planet.radius < ROTATION_RADIUS_LIMIT


def shot_crosses_sun(src: PlanetState, angle: float, tgt: PlanetState) -> bool:
    start_x = src.x + math.cos(angle) * (src.radius + PLANET_LAUNCH_RADIUS_OFFSET)
    start_y = src.y + math.sin(angle) * (src.radius + PLANET_LAUNCH_RADIUS_OFFSET)
    return point_to_segment_distance(BOARD_CENTER, (start_x, start_y), (tgt.x, tgt.y)) < SUN_RADIUS


def point_to_segment_distance(point: tuple[float, float], start: tuple[float, float], end: tuple[float, float]) -> float:
    segment_len_sq = (start[0] - end[0]) ** 2 + (start[1] - end[1]) ** 2
    if segment_len_sq == 0.0:
        return math.hypot(point[0] - start[0], point[1] - start[1])
    projection = (
        ((point[0] - start[0]) * (end[0] - start[0]) + (point[1] - start[1]) * (end[1] - start[1]))
        / segment_len_sq
    )
    projection = max(0.0, min(1.0, projection))
    closest_x = start[0] + projection * (end[0] - start[0])
    closest_y = start[1] + projection * (end[1] - start[1])
    return math.hypot(point[0] - closest_x, point[1] - closest_y)


Overwriting src/features.py


In [28]:
%%writefile src/game_types.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Any


@dataclass(slots=True)
class PlanetState:
    id: int
    owner: int
    x: float
    y: float
    radius: float
    ships: int
    production: int


@dataclass(slots=True)
class FleetState:
    id: int
    owner: int
    x: float
    y: float
    angle: float
    from_planet_id: int
    ships: int


@dataclass(slots=True)
class GameState:
    step: int
    player: int
    planets: list[PlanetState]
    fleets: list[FleetState]


def parse_observation(observation: Any) -> GameState:
    def obs_get(key: str, default: Any) -> Any:
        if isinstance(observation, dict):
            return observation.get(key, default)
        return getattr(observation, key, default)

    planets = [
        PlanetState(
            id=int(row[0]),
            owner=int(row[1]),
            x=float(row[2]),
            y=float(row[3]),
            radius=float(row[4]),
            ships=int(row[5]),
            production=int(row[6]),
        )
        for row in obs_get("planets", [])
    ]
    fleets = [
        FleetState(
            id=int(row[0]),
            owner=int(row[1]),
            x=float(row[2]),
            y=float(row[3]),
            angle=float(row[4]),
            from_planet_id=int(row[5]),
            ships=int(row[6]),
        )
        for row in obs_get("fleets", [])
    ]
    return GameState(
        step=int(obs_get("step", 0)),
        player=int(obs_get("player", 0)),
        planets=planets,
        fleets=fleets,
    )

Overwriting src/game_types.py


In [29]:
%%writefile src/policy.py

from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn


@dataclass(slots=True)
class PolicyOutput:
    target_logits: torch.Tensor
    value: torch.Tensor


class PlanetPolicy(nn.Module):
    def __init__(
        self,
        self_dim: int,
        candidate_dim: int,
        global_dim: int,
        candidate_count: int,
        hidden_size: int = 128,
    ) -> None:
        super().__init__()
        self.candidate_count = candidate_count
        self.self_encoder = nn.Sequential(
            nn.Linear(self_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )
        self.global_encoder = nn.Sequential(
            nn.Linear(global_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )
        self.candidate_encoder = nn.Sequential(
            nn.Linear(candidate_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )
        self.target_head = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )
        self.value_head = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )

    def forward(
        self,
        self_features: torch.Tensor,
        candidate_features: torch.Tensor,
        global_features: torch.Tensor,
        candidate_mask: torch.Tensor,
    ) -> PolicyOutput:
        self_hidden = self.self_encoder(self_features)
        global_hidden = self.global_encoder(global_features)
        candidate_hidden = self.candidate_encoder(candidate_features)
        expanded_self = self_hidden.unsqueeze(1).expand(-1, self.candidate_count, -1)
        expanded_global = global_hidden.unsqueeze(1).expand(-1, self.candidate_count, -1)
        joint = torch.cat([expanded_self, expanded_global, candidate_hidden], dim=-1)
        target_logits = self.target_head(joint).squeeze(-1)
        target_logits = target_logits.masked_fill(~candidate_mask, torch.finfo(target_logits.dtype).min)
        pooled_candidates = candidate_hidden.mean(dim=1)
        value = self.value_head(torch.cat([self_hidden, global_hidden, pooled_candidates], dim=-1)).squeeze(-1)
        return PolicyOutput(target_logits=target_logits, value=value)


Overwriting src/policy.py


In [30]:
%%writefile src/ppo.py

from __future__ import annotations

from dataclasses import dataclass

import torch
from torch.distributions import Categorical

from .policy import PolicyOutput


@dataclass(slots=True)
class SampledAction:
    target_index: torch.Tensor
    log_prob: torch.Tensor
    entropy: torch.Tensor


@dataclass(slots=True)
class TransitionBatch:
    self_features: torch.Tensor
    candidate_features: torch.Tensor
    global_features: torch.Tensor
    candidate_mask: torch.Tensor
    target_index: torch.Tensor
    log_prob: torch.Tensor
    returns: torch.Tensor
    advantages: torch.Tensor


def sample_actions(outputs: PolicyOutput, deterministic: bool) -> SampledAction:
    target_logits = safe_target_logits(outputs.target_logits)
    target_dist = Categorical(logits=target_logits)
    target_index = target_logits.argmax(dim=-1) if deterministic else target_dist.sample()

    log_prob, entropy = action_log_prob_and_entropy(outputs=outputs, target_index=target_index)
    return SampledAction(target_index=target_index, log_prob=log_prob, entropy=entropy)


def action_log_prob_and_entropy(
    outputs: PolicyOutput,
    target_index: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    target_logits = safe_target_logits(outputs.target_logits)
    target_dist = Categorical(logits=target_logits)
    target_log_prob = target_dist.log_prob(target_index)
    target_entropy = target_dist.entropy()
    return target_log_prob, target_entropy


def safe_target_logits(target_logits: torch.Tensor) -> torch.Tensor:
    invalid_rows = ~torch.isfinite(target_logits).any(dim=-1)
    if not invalid_rows.any():
        return target_logits
    safe_logits = target_logits.clone()
    safe_logits[invalid_rows, 0] = 0.0
    return safe_logits


def ppo_update(
    policy: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    batch: TransitionBatch,
    *,
    clip_coef: float,
    ent_coef: float,
    vf_coef: float,
    max_grad_norm: float,
    epochs: int,
    minibatch_size: int,
    device: torch.device,
) -> dict[str, float]:
    if batch.self_features.shape[0] == 0:
        return {"loss": 0.0, "policy_loss": 0.0, "value_loss": 0.0, "entropy": 0.0}
    self_features = batch.self_features.to(device)
    candidate_features = batch.candidate_features.to(device)
    global_features = batch.global_features.to(device)
    candidate_mask = batch.candidate_mask.to(device).bool()
    old_log_prob = batch.log_prob.to(device)
    target_index = batch.target_index.to(device)
    returns = batch.returns.to(device)
    advantages = batch.advantages.to(device)
    advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)
    size = self_features.shape[0]
    minibatch_size = min(size, max(1, minibatch_size))
    metrics = {"loss": 0.0, "policy_loss": 0.0, "value_loss": 0.0, "entropy": 0.0}
    updates = 0
    for _ in range(epochs):
        order = torch.randperm(size, device=device)
        for start in range(0, size, minibatch_size):
            idx = order[start : start + minibatch_size]
            outputs = policy(
                self_features[idx],
                candidate_features[idx],
                global_features[idx],
                candidate_mask[idx],
            )
            new_log_prob, entropy = action_log_prob_and_entropy(
                outputs,
                target_index[idx],
            )
            ratio = (new_log_prob - old_log_prob[idx]).exp()
            policy_loss = torch.maximum(
                -advantages[idx] * ratio,
                -advantages[idx] * torch.clamp(ratio, 1.0 - clip_coef, 1.0 + clip_coef),
            ).mean()
            value_loss = 0.5 * (returns[idx] - outputs.value).pow(2).mean()
            entropy_mean = entropy.mean()
            loss = policy_loss + vf_coef * value_loss - ent_coef * entropy_mean
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)
            optimizer.step()
            metrics["loss"] += float(loss.detach().cpu())
            metrics["policy_loss"] += float(policy_loss.detach().cpu())
            metrics["value_loss"] += float(value_loss.detach().cpu())
            metrics["entropy"] += float(entropy_mean.detach().cpu())
            updates += 1
    return {key: value / max(updates, 1) for key, value in metrics.items()}


Overwriting src/ppo.py


In [31]:
%%writefile src/opponents.py

from __future__ import annotations

from typing import Any, Protocol

import torch

from .config import TrainConfig
from .features import encode_turn
from .policy import PlanetPolicy
from .ppo import sample_actions


class OpponentPolicy(Protocol):
    def act(self, observation: Any) -> list[list[float | int]]:
        ...


class KaggleRandomOpponent:
    def __init__(self) -> None:
        from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent

        self._agent = random_agent

    def act(self, observation: Any) -> list[list[float | int]]:
        payload = {
            "player": obs_get(observation, "player", 0),
            "planets": list(obs_get(observation, "planets", [])),
        }
        return list(self._agent(payload))


class SelfPlayOpponent:
    def __init__(self, cfg: TrainConfig, device: torch.device, deterministic: bool = True) -> None:
        from .features import candidate_feature_dim, global_feature_dim, self_feature_dim

        self.cfg = cfg
        self.device = device
        self.deterministic = deterministic
        self.policy = PlanetPolicy(
            self_dim=self_feature_dim(),
            candidate_dim=candidate_feature_dim(),
            global_dim=global_feature_dim(),
            candidate_count=cfg.env.candidate_count,
            hidden_size=cfg.model.hidden_size,
        ).to(device)
        self.policy.eval()

    def sync_from(self, source_policy: PlanetPolicy) -> None:
        self.policy.load_state_dict(source_policy.state_dict())
        self.policy.eval()

    def act(self, observation: Any) -> list[list[float | int]]:
        batch = encode_turn(observation, self.cfg.env, env_index=0)
        if batch.self_features.shape[0] == 0:
            return []
        with torch.inference_mode():
            outputs = self.policy(
                torch.from_numpy(batch.self_features).to(self.device),
                torch.from_numpy(batch.candidate_features).to(self.device),
                torch.from_numpy(batch.global_features).to(self.device),
                torch.from_numpy(batch.candidate_mask).to(self.device).bool(),
            )
            sampled = sample_actions(outputs, deterministic=self.deterministic)
        target_indices = sampled.target_index.detach().cpu().numpy()
        moves: list[list[float | int]] = []
        for row_idx, context in enumerate(batch.contexts):
            target_idx = int(target_indices[row_idx])
            if target_idx == 0:
                continue
            if target_idx >= len(context.candidate_ids):
                continue
            if not context.candidate_mask[target_idx]:
                continue
            ships = int(context.ship_counts[target_idx])
            if ships <= 0:
                continue
            moves.append([context.source_id, float(context.target_angles[target_idx]), ships])
        return moves


def build_opponent(
    name: str,
    cfg: TrainConfig | None = None,
    device: torch.device | None = None,
) -> OpponentPolicy:
    if name == "random":
        return KaggleRandomOpponent()
    if name == "self":
        if cfg is None or device is None:
            raise ValueError("cfg and device are required for self opponent")
        return SelfPlayOpponent(cfg, device=device, deterministic=cfg.self_play_deterministic)
    raise ValueError(f"Unknown opponent: {name}")


def obs_get(observation: Any, key: str, default: Any) -> Any:
    if isinstance(observation, dict):
        return observation.get(key, default)
    return getattr(observation, key, default)


Overwriting src/opponents.py


In [32]:
%%writefile src/env.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from .config import TrainConfig
from .features import TurnBatch, encode_turn
from .opponents import OpponentPolicy


@dataclass(slots=True)
class StepResult:
    batch: TurnBatch
    reward: float
    done: bool
    info: dict[str, Any]


class OrbitWarsEnv:
    def __init__(
        self,
        cfg: TrainConfig,
        opponent: OpponentPolicy,
        make_fn: Any | None = None,
        env_index: int = 0,
    ) -> None:
        self.cfg = cfg
        self.opponent = opponent
        self.make_fn = make_fn
        self.env_index = env_index
        self.env: Any | None = None
        self.last_obs: Any | None = None
        self.last_opp_obs: Any | None = None
        self.episode_index = 0
        self.learner_player = 0

    def reset(self, seed: int | None = None) -> TurnBatch:
        make_fn = self.make_fn or default_make_fn()
        configuration: dict[str, Any] = {}
        if seed is not None:
            configuration["seed"] = int(seed)
            configuration["randomSeed"] = int(seed)
        if self.cfg.alternate_player_sides:
            self.learner_player = (self.env_index + self.episode_index) % 2
        else:
            self.learner_player = 0
        self.env = make_fn("orbit_wars", configuration=configuration, debug=False)
        self.env.reset(num_agents=2)
        states = self.env.step([[], []])
        learner_state = states[self.learner_player]
        opponent_state = states[1 - self.learner_player]
        self.last_obs = extract_observation(learner_state)
        self.last_opp_obs = extract_observation(opponent_state)
        self.episode_index += 1
        return encode_turn(self.last_obs, self.cfg.env, env_index=self.env_index)

    def step(self, player_action: list[list[float | int]]) -> StepResult:
        if self.env is None:
            raise RuntimeError("Call reset() before step().")
        opponent_action = self.opponent.act(self.last_opp_obs)
        if self.learner_player == 0:
            joint_action = [player_action, opponent_action]
        else:
            joint_action = [opponent_action, player_action]
        states = self.env.step(joint_action)
        player_state = states[self.learner_player]
        opp_state = states[1 - self.learner_player]
        self.last_obs = extract_observation(player_state)
        self.last_opp_obs = extract_observation(opp_state)
        done = extract_status(player_state) != "ACTIVE"
        reward = terminal_reward(player_state, opp_state) if done else 0.0
        batch = encode_turn(self.last_obs, self.cfg.env, env_index=self.env_index)
        info = {
            "learner_player": self.learner_player,
            "player_status": extract_status(player_state),
            "opponent_status": extract_status(opp_state),
            "reward": reward,
        }
        return StepResult(batch=batch, reward=reward, done=done, info=info)


def default_make_fn() -> Any:
    from kaggle_environments import make

    return make


def extract_observation(state: Any) -> Any:
    if isinstance(state, dict):
        return state.get("observation")
    return getattr(state, "observation")


def extract_status(state: Any) -> str:
    if isinstance(state, dict):
        return str(state.get("status", "UNKNOWN"))
    return str(getattr(state, "status", "UNKNOWN"))


def extract_reward(state: Any) -> float:
    if isinstance(state, dict):
        value = state.get("reward", 0.0)
    else:
        value = getattr(state, "reward", 0.0)
    return 0.0 if value is None else float(value)


def terminal_reward(player_state: Any, opp_state: Any) -> float:
    player_reward = extract_reward(player_state)
    opponent_reward = extract_reward(opp_state)
    if player_reward > 0.0 and opponent_reward > 0.0:
        return 0.0
    return player_reward


Overwriting src/env.py


In [33]:
%%writefile src/train.py

from __future__ import annotations

import argparse
import random
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch

from .config import TrainConfig, default_train_config_path, load_train_config
from .env import OrbitWarsEnv
from .features import TurnBatch, candidate_feature_dim, global_feature_dim, self_feature_dim
from .game_types import PlanetState
from .opponents import SelfPlayOpponent, build_opponent
from .policy import PlanetPolicy
from .ppo import TransitionBatch, ppo_update, sample_actions


@dataclass(slots=True)
class StepGroup:
    indices: list[int]
    reward: float
    done: bool


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, default=str(default_train_config_path()))
    return parser.parse_args()


def resolve_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def collect_rollout(
    envs: list[OrbitWarsEnv],
    batches: list[TurnBatch],
    policy: PlanetPolicy,
    cfg: TrainConfig,
    device: torch.device,
    next_seed: int,
) -> tuple[TransitionBatch, list[TurnBatch], int, dict[str, float]]:
    num_envs = len(envs)
    rollout_steps = cfg.ppo.rollout_steps
    gamma = cfg.ppo.gamma

    # Инференс размерностей из первого батча
    init = batches[0]
    self_dim = init.self_features.shape[-1]
    cand_dim_1 = init.candidate_features.shape[-1]
    glob_dim = init.global_features.shape[-1]

    # Безопасная начальная ёмкость: x2 от текущего размера rollout_steps
    base_per_step = sum(b.self_features.shape[0] for b in batches)
    capacity = max(int(base_per_step * rollout_steps * 1.5), 1024)

    # Преаллокация буферов
    self_buf = np.empty((capacity, self_dim), dtype=np.float32)
    cand_buf = np.empty((capacity, cfg.env.candidate_count, cand_dim_1), dtype=np.float32)
    glob_buf = np.empty((capacity, glob_dim), dtype=np.float32)
    mask_buf = np.empty((capacity, cfg.env.candidate_count), dtype=bool)
    val_buf = np.empty(capacity, dtype=np.float32)
    tgt_buf = np.empty(capacity, dtype=np.int64)
    lp_buf = np.empty(capacity, dtype=np.float32)

    ptr = 0
    groups_per_env: list[list[StepGroup]] = [[] for _ in range(num_envs)]
    running_episode_rewards = [0.0] * num_envs
    episode_rewards = []

    for _ in range(rollout_steps):
        offsets = np.cumsum([0] + [batch.self_features.shape[0] for batch in batches[:-1]])
        merged = merge_batches(batches)
        num_transitions = merged.self_features.shape[0]

        row_values = np.zeros(num_transitions, dtype=np.float32)
        sampled_target_index = np.zeros(num_transitions, dtype=np.int64)
        sampled_log_prob = np.zeros(num_transitions, dtype=np.float32)

        if num_transitions > 0:
            with torch.inference_mode():
                outputs = policy(
                    torch.from_numpy(merged.self_features).to(device, non_blocking=True),
                    torch.from_numpy(merged.candidate_features).to(device, non_blocking=True),
                    torch.from_numpy(merged.global_features).to(device, non_blocking=True),
                    torch.from_numpy(merged.candidate_mask).to(device, non_blocking=True).bool(),
                )
                sampled = sample_actions(outputs, deterministic=False)
                row_values = outputs.value.detach().cpu().numpy()
                sampled_target_index = sampled.target_index.detach().cpu().numpy()
                sampled_log_prob = sampled.log_prob.detach().cpu().numpy()

        next_batches: list[TurnBatch] = [None] * num_envs

        for env_idx in range(num_envs):
            batch = batches[env_idx]
            start = int(offsets[env_idx])
            num_contexts = batch.self_features.shape[0]
            moves = []
            group_indices = []

            for local_idx in range(num_contexts):
                global_idx = start + local_idx

                # ✅ Автоматическое расширение, если буфер заполнен
                if ptr >= self_buf.shape[0]:
                    new_cap = int(self_buf.shape[0] * 1.5)
                    self_buf.resize((new_cap, self_dim), refcheck=False)
                    cand_buf.resize((new_cap, cfg.env.candidate_count, cand_dim_1), refcheck=False)
                    glob_buf.resize((new_cap, glob_dim), refcheck=False)
                    mask_buf.resize((new_cap, cfg.env.candidate_count), refcheck=False)
                    val_buf.resize(new_cap, refcheck=False)
                    tgt_buf.resize(new_cap, refcheck=False)
                    lp_buf.resize(new_cap, refcheck=False)

                self_buf[ptr] = batch.self_features[local_idx]
                cand_buf[ptr] = batch.candidate_features[local_idx]
                glob_buf[ptr] = batch.global_features[local_idx]
                mask_buf[ptr] = batch.candidate_mask[local_idx]
                val_buf[ptr] = row_values[global_idx]
                tgt_buf[ptr] = sampled_target_index[global_idx] if num_transitions > 0 else 0
                lp_buf[ptr] = sampled_log_prob[global_idx] if num_transitions > 0 else 0.0

                context = batch.contexts[local_idx]
                is_valid_send = (
                    0 < tgt_buf[ptr] < len(context.candidate_ids)
                    and bool(context.candidate_mask[tgt_buf[ptr]])
                    and int(context.ship_counts[tgt_buf[ptr]]) > 0
                )

                group_indices.append(ptr)
                ptr += 1

                if is_valid_send:
                    ships = int(context.ship_counts[tgt_buf[ptr - 1]])
                    src_planet = find_planet(batch.state.planets, context.source_id)
                    if src_planet is not None and src_planet.ships >= ships:
                        moves.append([context.source_id, float(context.target_angles[tgt_buf[ptr - 1]]), ships])

            result = envs[env_idx].step(moves)
            reward_val = float(result.reward)
            running_episode_rewards[env_idx] += reward_val
            groups_per_env[env_idx].append(StepGroup(indices=group_indices, reward=reward_val, done=result.done))

            if result.done:
                episode_rewards.append(running_episode_rewards[env_idx])
                running_episode_rewards[env_idx] = 0.0
                next_seed += 1
                next_batches[env_idx] = envs[env_idx].reset(seed=next_seed)
            else:
                next_batches[env_idx] = result.batch

        batches = next_batches

    # GAE и Returns
    total_transitions = ptr
    returns = np.zeros(total_transitions, dtype=np.float32)
    advantages = np.zeros(total_transitions, dtype=np.float32)
    next_state_values = bootstrap_values(policy, batches, device)

    idx = 0
    for env_idx in range(num_envs):
        future_return = next_state_values[env_idx]
        for group in reversed(groups_per_env[env_idx]):
            future_return = group.reward + gamma * future_return * (1.0 - float(group.done))
            end = idx + len(group.indices)
            returns[idx:end] = future_return
            advantages[idx:end] = future_return - val_buf[idx:end]
            idx = end

    batch = TransitionBatch(
        self_features=torch.from_numpy(self_buf[:total_transitions]),
        candidate_features=torch.from_numpy(cand_buf[:total_transitions]),
        global_features=torch.from_numpy(glob_buf[:total_transitions]),
        candidate_mask=torch.from_numpy(mask_buf[:total_transitions]),
        target_index=torch.from_numpy(tgt_buf[:total_transitions]),
        log_prob=torch.from_numpy(lp_buf[:total_transitions]),
        returns=torch.from_numpy(returns),
        advantages=torch.from_numpy(advantages),
    )

    stats = {
        "episode_reward_mean": float(np.mean(episode_rewards)) if episode_rewards else 0.0,
        "episodes_finished": float(len(episode_rewards)),
        "samples": float(total_transitions),
    }
    return batch, batches, next_seed, stats


def bootstrap_values(policy: PlanetPolicy, batches: list[TurnBatch], device: torch.device) -> list[float]:
    merged = merge_batches(batches)
    if merged.self_features.shape[0] == 0:
        return [0.0 for _ in batches]
    offsets = np.cumsum([0] + [batch.self_features.shape[0] for batch in batches[:-1]])
    with torch.inference_mode():
        outputs = policy(
            torch.from_numpy(merged.self_features).to(device),
            torch.from_numpy(merged.candidate_features).to(device),
            torch.from_numpy(merged.global_features).to(device),
            torch.from_numpy(merged.candidate_mask).to(device).bool(),
        )
    values = outputs.value.detach().cpu().numpy()
    per_env = []
    for env_idx, batch in enumerate(batches):
        start = int(offsets[env_idx])
        count = batch.self_features.shape[0]
        per_env.append(0.0 if count == 0 else float(values[start : start + count].mean()))
    return per_env


def merge_batches(batches: list[TurnBatch]) -> TurnBatch:
    if not batches:
        raise ValueError("batches must not be empty")
    has_rows = any(batch.self_features.shape[0] > 0 for batch in batches)
    self_rows = (
        np.concatenate([batch.self_features for batch in batches], axis=0)
        if has_rows
        else np.zeros((0, self_feature_dim()), dtype=np.float32)
    )
    candidate_rows = (
        np.concatenate([batch.candidate_features for batch in batches], axis=0)
        if has_rows
        else np.zeros((0, batches[0].candidate_features.shape[1], candidate_feature_dim()), dtype=np.float32)
    )
    global_rows = (
        np.concatenate([batch.global_features for batch in batches], axis=0)
        if has_rows
        else np.zeros((0, global_feature_dim()), dtype=np.float32)
    )
    candidate_masks = (
        np.concatenate([batch.candidate_mask for batch in batches], axis=0)
        if has_rows
        else np.zeros((0, batches[0].candidate_mask.shape[1]), dtype=bool)
    )
    return TurnBatch(
        self_features=self_rows,
        candidate_features=candidate_rows,
        global_features=global_rows,
        candidate_mask=candidate_masks,
        contexts=[context for batch in batches for context in batch.contexts],
        state=batches[0].state,
    )


def save_checkpoint(
    save_dir: Path,
    run_name: str,
    update: int,
    policy: PlanetPolicy,
    optimizer: torch.optim.Optimizer,
    cfg: TrainConfig,
) -> None:
    run_dir = save_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "update": update,
            "policy": policy.state_dict(),
            "optimizer": optimizer.state_dict(),
            "config": cfg,
        },
        run_dir / "ckpt_last.pt",
    )
    torch.save(
        {
            "update": update,
            "policy": policy.state_dict(),
            "optimizer": optimizer.state_dict(),
            "config": cfg,
        },
        run_dir / f"ckpt_{update:06d}.pt",
    )


def find_planet(planets: list[PlanetState], planet_id: int) -> PlanetState | None:
    for planet in planets:
        if planet.id == planet_id:
            return planet
    return None


def main() -> None:
    args = parse_args()
    cfg = load_train_config(args.config)
    seed_everything(cfg.seed)
    device = resolve_device(cfg.device)
    opponent = build_opponent(cfg.opponent, cfg=cfg, device=device)
    envs = [OrbitWarsEnv(cfg, opponent, env_index=idx) for idx in range(cfg.ppo.num_envs)]
    next_seed = cfg.seed
    batches = []
    for env in envs:
        batches.append(env.reset(seed=next_seed))
        next_seed += 1
    policy = PlanetPolicy(
        self_dim=self_feature_dim(),
        candidate_dim=candidate_feature_dim(),
        global_dim=global_feature_dim(),
        candidate_count=cfg.env.candidate_count,
        hidden_size=cfg.model.hidden_size,
    ).to(device)
    if isinstance(opponent, SelfPlayOpponent):
        opponent.sync_from(policy)
    optimizer = torch.optim.Adam(policy.parameters(), lr=cfg.ppo.lr)
    save_dir = Path(cfg.save_dir)
    for update in range(1, cfg.ppo.total_updates + 1):
        batch, batches, next_seed, stats = collect_rollout(envs, batches, policy, cfg, device, next_seed)
        metrics = ppo_update(
            policy,
            optimizer,
            batch,
            clip_coef=cfg.ppo.clip_coef,
            ent_coef=cfg.ppo.ent_coef,
            vf_coef=cfg.ppo.vf_coef,
            max_grad_norm=cfg.ppo.max_grad_norm,
            epochs=cfg.ppo.epochs,
            minibatch_size=cfg.ppo.minibatch_size,
            device=device,
        )
        if isinstance(opponent, SelfPlayOpponent) and update % cfg.self_play_update_interval == 0:
            opponent.sync_from(policy)
        if update % cfg.log_every == 0:
            print(
                f"update={update} episode_reward_mean={stats['episode_reward_mean']:.4f} "
                f"episodes={int(stats['episodes_finished'])} samples={int(stats['samples'])} "
                f"loss={metrics['loss']:.4f}"
            )
        if update % cfg.checkpoint_every == 0 or update == cfg.ppo.total_updates:
            save_checkpoint(save_dir, cfg.run_name, update, policy, optimizer, cfg)


if __name__ == "__main__":
    main()


Overwriting src/train.py


In [34]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [35]:
!python -m src.train --config default_cfg.yaml

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 23.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_s

In [36]:
%%writefile decode_weights.py
# auto-generated: decodes the embedded base64 MLP weights into weights.npz
import base64, pathlib

import torch
import numpy as np

# 1. Load the PyTorch file (.pt)
# Set weights_only=True for security if you're loading a state_dict
data = torch.load('artifacts/orbit_wars_ppo/ckpt_last.pt', map_location="cpu", weights_only=False)

# 2. Convert Tensors to NumPy arrays
# If the data is a dictionary (common for state_dicts)
if isinstance(data, dict):
    numpy_data = {k: v.cpu().numpy() if torch.is_tensor(v) else v for k, v in data.items()}
    # 3. Save as .npz
    np.savez('model_weights.npz', **numpy_data)
else:
    # If the data is a single tensor
    numpy_data = data.cpu().numpy()
    np.savez('model_weights.npz', my_array=numpy_data)

print("Conversion complete: model_weights.npz saved.")

Overwriting decode_weights.py


In [37]:
!python decode_weights.py


Conversion complete: model_weights.npz saved.


## 5. The agent

A single self-contained `submission.py`. Everything is pure stdlib + numpy — no torch,
no extra dependencies. The rule-base body (~3300 lines) is inlined verbatim and its entry
function is renamed to `_v4_agent_internal`; our `agent(obs, config)` wraps it and applies
the validator.

Layout of the file:

1. Imports + the numpy `_NumpyValidator` class (forward only)
2. `_encode_shot_np(...)` — 24-dim feature builder
3. `_find_target_ray(...)` — rebuild target id from `(src, angle)` via ray projection
4. The full inlined rule-base agent (renamed to `_v4_agent_internal`)
5. `agent(obs, config)` — calls `_v4_agent_internal`, then drops shots below threshold


In [38]:
%%writefile -a main.py
"""Hybrid agent: v4 + Shot Validator override.
Auto-generated in notebook. Do not edit manually.
"""
import math as _math_hybrid
import os as _os_hybrid
from pathlib import Path as _Path_hybrid
import numpy as _np_hybrid

_VAL_THRESHOLD = 0.3000  # Change this value directly if needed

# ---- Shot Validator (numpy) ----
def _find_weights_path():
    candidates = [
        _Path_hybrid("/kaggle_simulations/agent/weights.npz"),
        _Path_hybrid.cwd() / "weights.npz",
        _Path_hybrid("weights.npz"),
    ]
    try:
        candidates.insert(0, _Path_hybrid(__file__).resolve().parent / "weights.npz")
    except NameError:
        pass
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

_WEIGHTS_PATH = _find_weights_path()

class _NumpyValidator:
    def __init__(self, w_path):
        npz = _np_hybrid.load(str(w_path))
        self.w0 = npz["w0"]; self.b0 = npz["b0"]
        self.w2 = npz["w2"]; self.b2 = npz["b2"]
        self.w4 = npz["w4"]; self.b4 = npz["b4"]
    def forward(self, x):
        h = _np_hybrid.maximum(0.0, x @ self.w0.T + self.b0)
        h = _np_hybrid.maximum(0.0, h @ self.w2.T + self.b2)
        return (h @ self.w4.T + self.b4).reshape(-1)
    def proba(self, x):
        z = self.forward(x)
        return 1.0 / (1.0 + _np_hybrid.exp(-z))

try:
    _VALIDATOR = _NumpyValidator(_WEIGHTS_PATH) if _WEIGHTS_PATH.exists() else None
except Exception:
    _VALIDATOR = None

_FEATURE_DIM = 24

def _encode_shot_np(obs, src_id, target_id, ships_sent):
    BOARD = 100.0; MAX_SPEED = 6.0
    pdict = {}
    for p in obs["planets"]:
        pid = int(p[0])
        pdict[pid] = (int(p[1]), float(p[2]), float(p[3]), float(p[4]), int(p[5]), float(p[6]))
    if src_id not in pdict or target_id not in pdict:
        return None
    src = pdict[src_id]; tgt = pdict[target_id]
    me = int(obs.get("player", 0))
    fleets = obs.get("fleets", [])
    planets = obs["planets"]
    my_ships_total = sum(int(p[5]) for p in planets if int(p[1]) == me)
    enemy_ships_total = sum(int(p[5]) for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    my_planets = sum(1 for p in planets if int(p[1]) == me)
    enemy_planets = sum(1 for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    src_owner, sx, sy, sr, ss, sp = src
    tgt_owner, tx, ty, tr, ts, tp = tgt
    dx = tx - sx; dy = ty - sy
    dist = max(_math_hybrid.hypot(dx, dy) - sr - tr, 0.0)
    if ships_sent <= 0:
        speed = 1.0
    else:
        speed = 1.0 + (MAX_SPEED - 1.0) * (_math_hybrid.log(max(ships_sent, 1)) / _math_hybrid.log(1000.0)) ** 1.5
    eta = dist / max(speed, 0.5)
    own_self = 1.0 if tgt_owner == me else 0.0
    own_neutral = 1.0 if tgt_owner < 0 else 0.0
    own_enemy = 1.0 if (tgt_owner >= 0 and tgt_owner != me) else 0.0
    ship_frac = ships_sent / max(ss, 1)
    ally_n = 0; ally_s = 0; enemy_n = 0; enemy_s = 0
    for f in fleets:
        owner = int(f[1]); shp = int(f[6])
        if owner == me:
            ally_n += 1; ally_s += shp
        else:
            enemy_n += 1; enemy_s += shp
    turn = int(obs.get("step", 0))
    feat = _np_hybrid.array([
        ss / 100.0, sp / 5.0, sr / 4.0,
        ts / 100.0, tp / 5.0, tr / 4.0,
        own_self, own_neutral, own_enemy,
        ships_sent / 100.0, ship_frac,
        dist / BOARD, eta / 60.0, speed / MAX_SPEED,
        ally_n / 10.0, ally_s / 100.0,
        enemy_n / 10.0, enemy_s / 100.0,
        turn / 500.0,
        my_ships_total / 200.0, enemy_ships_total / 200.0,
        (my_ships_total - enemy_ships_total) / 200.0,
        my_planets / 20.0, enemy_planets / 20.0,
    ], dtype=_np_hybrid.float32)
    return feat

def _find_target_ray(src_xy, send_angle, planets, ray_horizon=200.0, perp_margin=1.0):
    sx, sy = src_xy
    fx = _math_hybrid.cos(send_angle); fy = _math_hybrid.sin(send_angle)
    best_pid = -1; best_perp = 1e9
    for p in planets:
        pid = int(p[0]); px = float(p[2]); py = float(p[3]); pr = float(p[4])
        dx = px - sx; dy = py - sy
        t = dx * fx + dy * fy
        if t <= 0 or t > ray_horizon:
            continue
        perp = abs(dx * fy - dy * fx)
        if perp <= pr + perp_margin and perp < best_perp:
            best_perp = perp; best_pid = pid
    return best_pid

Appending to main.py


In [39]:
import re
from pathlib import Path

# 👉 Укажите правильный путь к вашему v4-агенту
V4_PATH = Path("/kaggle/input/datasets/thisisn0mad/orbit-wars-rule-based-submission-by-rudra/rule-based submission.py")
assert V4_PATH.exists(), f"V4 source not found at {V4_PATH}"

v4_code = V4_PATH.read_text()
# Rename entry point & remove __all__ to avoid conflicts
v4_code = re.sub(r'^def agent\(obs', 'def _v4_agent_internal(obs', v4_code, flags=re.MULTILINE)
v4_code = re.sub(r'^__all__\s*=.*$', '', v4_code, flags=re.MULTILINE)

with open("main.py", "a") as f:
    f.write("\n\n# ---- v4 source (inlined below) ----\n")
    f.write(v4_code)
print("✅ V4 agent injected & renamed successfully.")

✅ V4 agent injected & renamed successfully.


In [40]:
%%writefile -a main.py

# ---- Hybrid entrypoint ----
def agent(obs, config=None):
    moves = _v4_agent_internal(obs)
    if not moves or _VALIDATOR is None:
        return moves
    side = int(obs.get("player", 0))
    planets = obs["planets"]
    owner_by_id = {}
    src_xy = {}
    for p in planets:
        pid = int(p[0])
        owner_by_id[pid] = int(p[1])
        src_xy[pid] = (float(p[2]), float(p[3]))
    feats = []
    idxs = []
    for i, mv in enumerate(moves):
        try:
            src_id = int(mv[0]); ang = float(mv[1]); ships = int(mv[2])
        except Exception:
            continue
        if src_id not in src_xy:
            continue
        tgt_id = _find_target_ray(src_xy[src_id], ang, planets)
        if tgt_id < 0 or tgt_id == src_id:
            continue
        if owner_by_id.get(tgt_id, -2) == side:
            continue  # own-planet reinforcement: always keep
        feat = _encode_shot_np(obs, src_id, tgt_id, ships)
        if feat is None:
            continue
        feats.append(feat); idxs.append(i)
    if not feats:
        return moves
    x = _np_hybrid.stack(feats)
    probs = _VALIDATOR.proba(x)
    keep = [True] * len(moves)
    for i, prob in zip(idxs, probs):
        if prob < _VAL_THRESHOLD:
            keep[i] = False
    return [mv for i, mv in enumerate(moves) if keep[i]]

__all__ = ["agent"]

Appending to main.py


## 6. Sanity check

Make sure `submission.py` parses cleanly and the agent function is callable.


In [41]:
# Sanity: ensure the submission imports cleanly with weights present
import importlib.util, pathlib
assert pathlib.Path("/kaggle/working/model_weights.npz").exists(), "weights.npz must exist before importing"
spec = importlib.util.spec_from_file_location("main", "main.py")
m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
print("agent:", m.agent)
print("validator loaded:", m._VALIDATOR is not None)
print("threshold:", m._VAL_THRESHOLD)


agent: <function agent at 0x7c94ca80cd60>
validator loaded: False
threshold: 0.3


## 7. Submit

To submit: download both `submission.py` and `weights.npz` from this notebook, package
them into a single tar.gz with both files at the **root** (no enclosing folder), and
upload via the *Submit Predictions* button on the competition page.

```bash
tar -czf submission.tar.gz submission.py weights.npz
# then upload submission.tar.gz
```

If you fork and re-run this notebook end to end, the two files will be sitting in the
working directory ready to download.


In [42]:
!tar -czf submission.tar.gz  main.py model_weights.npz

In [43]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [44]:
from kaggle_environments import make

# Test it against the random agent
env2 = make("orbit_wars", debug=True)
from main import *
env2.run([agent, "random"])



final = env2.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env2.render(mode="ipython", width=800, height=600)

Player 0: reward=1, status=DONE
Player 1: reward=-1, status=DONE


## 8. What did *not* work (so you don't repeat it)

Five separate ML directions before this hybrid hit the same tier3+ wall:

- **PPO from random init** with a structured-policy network — collapsed to no-op around
  update ~80, classic dense-shaping trap.
- **PPO with curriculum** (sniper → tier1 → tier2 → tier3 → mixed) — survived longer but
  still 0% on tier3 in eval.
- **PPO with smoother 5-phase curriculum + lower lr + lower shaping coef** — entropy stayed
  healthy through training, still 0% on tier3.
- **SFT (single teacher, orbitbotnext)** — got to 75% vs sniper but tier3+ still 0%.
- **SFT (multi-teacher: orbitbotnext + v4 + exp004_a)** — val pos_acc went *up* (20% → 34%)
  but in-game wr collapsed to **2% overall** (sniper 12%, everything else 0%). Conflicting
  teacher labels averaged out into a policy that hesitated everywhere.

The lesson that pushed us toward the validator hybrid: when the rule-base ceiling and the
ML floor don't intersect, don't try to make ML stand alone — let it *edit* the rule-base.

A few smaller dead ends:

- **Hand-tuning the rule-base constants** (HOSTILE_REINFORCE, attack-cost, rotating-opening
  thresholds): every change that helped against one opponent hurt against another. The
  rule-base has already absorbed the easy wins from constant tuning.
- **Threshold 0.5 for the validator**: rejects too many shots, hybrid wr drops to 57%.
  More-rejection is not always better.


## 9. Acknowledgements

- [Pilkwang Kim](https://www.kaggle.com/pilkwang) — `Orbit Wars: Structured Baseline` (the common ancestor of the rule-base used here)
- [Roman Tamrazov](https://www.kaggle.com/romantamrazov) — public structured-baseline derivative
- [Yegor Khnykin](https://www.kaggle.com/ykhnkf) — public hybrid lineage
- [Kaggle Simulations](https://www.kaggle.com/c/orbit-wars) — for the competition format that allows quick local benchmarking against forks of public agents

If this notebook saves you a few days of PPO debugging, please consider an upvote 🛰️
